# Data Embedding

## Content -> Vectors

An **embedding** maps text (a chunk) to a fixed-size vector of floats: `text -> [0.12, -0.55, 0.87, ...]`.

- Semantically similar texts produce **similar vectors** (near in vector space).
- Output dimension is fixed per model: e.g. 384 (many sentence-transformers), 768 (BGE), 1536 (OpenAI `text-embedding-3-small`).
- Feeds the retrieval step: vector stores search by **cosine / dot-product similarity**.

## The `Embeddings` Interface (`langchain_core.embeddings`)

Two key methods:

- `embed_documents([chunk1, chunk2, ...])` -> `list[list[float]]` - for storing/chunking
- `embed_query("user question")` -> `list[float]` - for queries

Providers implement the same interface, so swapping models changes one line.

## Common Providers

- `OpenAIEmbeddings` - API key, hosted
- `HuggingFaceEmbeddings` / `SentenceTransformerEmbeddings` - local, model by name
- `OllamaEmbeddings` - local via Ollama, e.g. `nomic-embed-text` (needs no key)

## Pipeline Position

`ingest -> split -> embed each chunk -> store vectors + text (vector store)`

Embedding is still *offline preprocessing*. Live retrieval later embeds the query with `embed_query` and vector-searches the stored chunk vectors.

In [ ]:

import os
from langchain_openai import OpenAIEmbeddings

# set the key from your env (never hardcode it)
os.environ["OPENAI_API_KEY"] = "sk-...your-key..."

# --- single query (for live retrieval) ---
emb = OpenAIEmbeddings(model="text-embedding-3-small")
query_vec = emb.embed_query("attention mechanism in transformers")
print("query vector length:", len(query_vec))          # 1536

# --- batch embed documents (for offline ingestion) ---
docs = ["chunk 1 of a paper", "chunk 2 of a paper", "chunk 3 of a paper"]
doc_vecs = emb.embed_documents(docs)
print("doc vectors:", len(doc_vecs), "each", len(doc_vecs[0]))

# same interface works with langchain_core.documents.Document objects
# from langchain_core.documents import Document
# emb.embed_documents([Document(page_content="...")])   # works the same way


In [ ]:

from langchain_ollama import OllamaEmbeddings

# --- 1. nomic-embed-text (default choice) ---
emb_nomic = OllamaEmbeddings(model="nomic-embed-text")
vec = emb_nomic.embed_query("what is attention mechanism?")
print("nomic-embed-text:", len(vec), "dims")
print("first 5 values:", [round(x, 4) for x in vec[:5]])


In [ ]:

# --- 2. compare all three embedding models ---
models = {
    "nomic-embed-text":        768,
    "all-minilm":              384,
    "snowflake-arctic-embed":  1024,
}

for model_name, expected_dims in models.items():
    emb = OllamaEmbeddings(model=model_name)
    vec = emb.embed_query("chunking documents for RAG pipelines")
    status = "ok" if len(vec) == expected_dims else f"mismatch ({len(vec)})"
    print(f"{model_name:28s} -> {len(vec):4d} dims  {status}")


In [ ]:

# --- 3. embed multiple document chunks (offline ingestion) ---
chunks = [
    "Jump into bed, plug in my phone when I still got ninety percent left to go",
    "And I fall asleep but before I do I sit in silence for an hour or two",
    "These vivid thoughts get into view and this what it sound like when I dream of you",
]

emb = OllamaEmbeddings(model="nomic-embed-text")
doc_vecs = emb.embed_documents(chunks)
print(f"embedded {len(doc_vecs)} chunks, each {len(doc_vecs[0])} dims")
for i, (chunk, vec) in enumerate(zip(chunks, doc_vecs)):
    print(f"  chunk {i}: {len(vec)} dims | head: {chunk[:45]}...")


In [ ]:

# --- 4. basic similarity check with cosine ---
import math

def cosine_sim(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * y for x, y in zip(b, b)))
    return dot / (na * nb) if na and nb else 0.0

emb = OllamaEmbeddings(model="nomic-embed-text")
v1 = emb.embed_query("what is attention mechanism")
v2 = emb.embed_query("attention mechanism explained")
v3 = emb.embed_query("random unrelated text about pasta")

print("same topic:      ", round(cosine_sim(v1, v2), 4))
print("different topic: ", round(cosine_sim(v1, v3), 4))


In [ ]:

from langchain_huggingface import HuggingFaceEmbeddings

# popular embedding models on HuggingFace
# - "all-MiniLM-L6-v2"       (384 dims, fastest, English-only)
# - "BAAI/bge-small-en-v1.5" (384 dims, good quality)
# - "BAAI/bge-large-en-v1.5" (1024 dims, stronger)
# - "BAAI/bge-m3"            (1024 dims, multilingual)

emb = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},    # "cuda" if you have a GPU
)

query_vec = emb.embed_query("hello world")
print("all-MiniLM-L6-v2:", len(query_vec), "dims")

doc_vecs = emb.embed_documents(["chunk one", "chunk two"])
print("batch embed:", len(doc_vecs), "docs")
